In [9]:
import pandas as pd

In [10]:
raw_df=pd.read_csv('insurance.csv')

In [11]:
raw_df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


In [12]:
inputs=raw_df[['age','sex','bmi','children','smoker','region']]
target=raw_df['charges']

In [13]:
numerical_cols=['age','bmi','children']

In [14]:
from sklearn.model_selection import train_test_split
inputs_train,input_test,target_train,target_test=train_test_split(inputs, target, test_size=0.25, random_state=42)

In [15]:
dict1={'no':0,'yes':1}

In [16]:
dict2={'male':1,'female':0}

In [17]:
dict3={'southeast':1,'southwest':2,'northeast':3,'northwest':4}

In [18]:
inputs_train['smoker']=inputs_train.smoker.map(dict1)
input_test['smoker']=input_test.smoker.map(dict1)

In [19]:
inputs_train['sex']=inputs_train.sex.map(dict2)
input_test['sex']=input_test.sex.map(dict2)

In [20]:
inputs_train['region']=inputs_train.region.map(dict3)
input_test['region']=input_test.region.map(dict3)

In [21]:
input_test

,age,sex,bmi,children,smoker,region
764,45,0,25.175,2,0,3
887,36,0,30.020,0,0,4
890,64,0,26.885,0,1,4
1293,46,1,25.745,3,0,4
259,19,1,31.920,0,1,4
...,...,...,...,...,...,...
342,60,0,27.550,0,0,3
308,58,1,34.865,0,0,3
1128,34,1,32.800,1,0,2
503,19,1,30.250,0,1,1


In [22]:
inputs_train

,age,sex,bmi,children,smoker,region
693,24,1,23.655,0,0,4
1297,28,0,26.510,2,0,1
634,51,1,39.700,1,0,2
1022,47,1,36.080,1,1,1
178,46,0,28.900,2,0,2
...,...,...,...,...,...,...
1095,18,0,31.350,4,0,3
1130,39,0,23.870,5,0,1
1294,58,1,25.175,0,0,3
860,37,0,47.600,2,1,2


In [36]:
from xgboost import XGBRegressor

In [38]:
model=XGBRegressor(random_state=42)

In [39]:
model.fit(inputs_train,target_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [40]:
preds=model.predict(inputs_train)

In [41]:
preds

array([ 2404.627 ,  3863.4333,  8968.704 , ..., 11674.673 , 45935.145 ,
       10893.275 ], dtype=float32)

In [42]:
from sklearn.metrics import r2_score

In [43]:
lossy=r2_score(target_train,preds)

In [44]:
lossy

0.9954040703345491

In [45]:
predict=model.predict(input_test)

In [46]:
lossy2=r2_score(target_test,predict)

In [47]:
lossy2

0.8348220378696097

In [50]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'n_estimators': [180, 200, 220],
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [5, 6, 7],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0.1, 0.2, 0.3],
    'reg_alpha': [0.01, 0.1, 1],
    'reg_lambda': [0.01, 0.1, 1]
}




In [52]:
# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
                           param_grid=param_grid, 
                           scoring='neg_mean_squared_error', 
                           cv=5, 
                           verbose=2, 
                           n_jobs=-1)

# Fit the Grid Search to the data
grid_search.fit(inputs_train,target_train)

Fitting 5 folds for each of 6561 candidates, totalling 32805 fits


KeyboardInterrupt: 